In [9]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
# data visualixxaton
import matplotlib.pyplot as plt

import cv2
AUTOTUNE = tf.data.AUTOTUNE


In [10]:
BATCH_SIZE = 80
IMAGE_SIZE = 256
CHANNELS=3
EPOCHS=50

In [11]:
dataDir = tf.keras.preprocessing.image_dataset_from_directory(
    r"C:\Users\YASHASH RAVI\Downloads\plantimages\PlantVillage",
    shuffle=True,
    image_size=(IMAGE_SIZE,IMAGE_SIZE),
    batch_size=BATCH_SIZE
)

print(dataDir)

# column name
column_name = dataDir.class_names

print(np.array(column_name))


Found 32681 files belonging to 15 classes.
<_PrefetchDataset element_spec=(TensorSpec(shape=(None, 256, 256, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int32, name=None))>
['Pepper__bell___Bacterial_spot' 'Pepper__bell___healthy'
 'Potato___Early_blight' 'Potato___Late_blight' 'Potato___healthy'
 'Tomato_Bacterial_spot' 'Tomato_Early_blight' 'Tomato_Late_blight'
 'Tomato_Leaf_Mold' 'Tomato_Septoria_leaf_spot'
 'Tomato_Spider_mites_Two_spotted_spider_mite' 'Tomato__Target_Spot'
 'Tomato__Tomato_YellowLeaf__Curl_Virus' 'Tomato__Tomato_mosaic_virus'
 'Tomato_healthy']


In [6]:
# train dataset
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    r"C:\Users\YASHASH RAVI\Downloads\plantimages\PlantVillage",
    validation_split=0.3,   # 30% reserved for val+test
    subset="training",
    seed=123,
    image_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
)


val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    r"C:\Users\YASHASH RAVI\Downloads\plantimages\PlantVillage",
    validation_split=0.3,   # same 30% reserved
    subset="validation",
    seed=123,
    image_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
)

val_batches = tf.data.experimental.cardinality(val_ds)
test_size = val_batches // 3   # roughly 10% of total dataset

test_ds = val_ds.take(test_size)       # first 10% → test
val_ds = val_ds.skip(test_size)        # remaining 20% → validation



Found 20638 files belonging to 15 classes.
Using 14447 files for training.
Found 20638 files belonging to 15 classes.
Using 6191 files for validation.


In [ ]:
# x = []
# y=  []

# for image_batch, label_batch in train_ds[]:
#     # for clear img convert to int
#     plt.imshow(image_batch[0].numpy().astype(int))
#     plt.axis("off")
#     plt.title(column_name[label_batch[0]])
#     plt.show()
#     x.append(image_batch[0].numpy().astype(int))
#     y.append(column_name[label_batch[0]])

In [1]:
class_names = dataDir.class_names
class_names

NameError: name 'dataDir' is not defined

In [ ]:
len(dataDir)

In [ ]:
# pre proceesing
data_augmentation = keras.Sequential([
    keras.layers.RandomFlip("horizontal_and_vertical"),
    keras.layers.RandomRotation(0.2),
])

resize_rescale = keras.Sequential([
    keras.layers.Resizing(IMAGE_SIZE,IMAGE_SIZE),
    keras.layers.Rescaling(1./255),
])

In [ ]:
input_shape = (BATCH_SIZE, IMAGE_SIZE, IMAGE_SIZE, CHANNELS)
n_classes = 3

model = keras.models.Sequential([
    resize_rescale,
    data_augmentation,
    keras.layers.Conv2D(32, kernel_size = (3,3), activation='relu', input_shape=input_shape),
    keras.layers.MaxPooling2D((2, 2)),
    keras.layers.Conv2D(64,  kernel_size = (3,3), activation='relu'),
    keras.layers.MaxPooling2D((2, 2)),
    keras.layers.Conv2D(64,  kernel_size = (3,3), activation='relu'),
    keras.layers.MaxPooling2D((2, 2)),
    keras.layers.Conv2D(64, (3, 3), activation='relu'),
    keras.layers.MaxPooling2D((2, 2)),
    keras.layers.Conv2D(64, (3, 3), activation='relu'),
    keras.layers.MaxPooling2D((2, 2)),
    keras.layers.Conv2D(64, (3, 3), activation='relu'),
    keras.layers.MaxPooling2D((2, 2)),
    keras.layers.Flatten(),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dense(len(column_name), activation='softmax'),
])

model.build(input_shape=input_shape)

In [ ]:
model.summary()

In [ ]:
# model.build(input_shape=(BATCH_SIZE,IMAGE_SIZE,IMAGE_SIZE,CHANNELS))

model.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=['accuracy']
)

In [ ]:
# from tensorflow.keras.callbacks import EarlyStopping

early_stop = keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

model.fit(train_ds,
          validation_data=val_ds,
          batch_size=80,
          epochs=30,
          callbacks=[early_stop]
          )


In [ ]:
model.evaluate(test_ds)

In [ ]:
model.save(r"E:\python\tenserflow\Exercises\potato_healthy_plant_detection\model_healty_plant\plant_status_detector_v1.5.keras")

In [ ]:
model.predict(test_ds)

In [ ]:
for i in test_ds.take(5):
    print(i)

In [ ]:
import numpy as np

y_pred = model.predict(test_ds)              # shape (2080, 15)
y_pred_classes = np.argmax(y_pred, axis=1)   # pick highest probability class


In [ ]:
y_true = np.concatenate([y.numpy() for x, y in test_ds], axis=0)


In [ ]:
from sklearn.metrics import accuracy_score

acc = accuracy_score(y_true, y_pred_classes)
print("Test Accuracy:", acc)


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(y_true, y_pred_classes))
print(confusion_matrix(y_true, y_pred_classes))


In [ ]:
print(model.predict(test_ds))

In [ ]:
print(confusion_matrix(y_true, y_pred_classes))

In [8]:
model_2 = keras.models.load_model(r"E:\python\tenserflow\Exercises\potato_healthy_plant_detection\model_healty_plant\plant_status_detector_v2.keras")

In [11]:
import numpy as np

y_pred = model_2.predict(test_ds)              # shape (2080, 15)
y_pred_classes = np.argmax(y_pred, axis=1)   # pick highest probability class

y_true = np.concatenate([y.numpy() for x, y in test_ds], axis=0)



26/26 ━━━━━━━━━━━━━━━━━━━━ 4s 162ms/step


In [12]:
from sklearn.metrics import accuracy_score

acc = accuracy_score(y_true, y_pred_classes)
print("Test Accuracy:", acc)

Test Accuracy: 0.08509615384615385


In [13]:
from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(y_true, y_pred_classes))
print(confusion_matrix(y_true, y_pred_classes))


              precision    recall  f1-score   support

           0       0.08      0.07      0.08       120
           1       0.05      0.06      0.06       141
           2       0.06      0.06      0.06       104
           3       0.03      0.03      0.03       100
           4       0.00      0.00      0.00        16
           5       0.11      0.11      0.11       215
           6       0.04      0.04      0.04        99
           7       0.11      0.11      0.11       216
           8       0.03      0.04      0.04        83
           9       0.09      0.09      0.09       182
          10       0.09      0.09      0.09       161
          11       0.05      0.05      0.05       147
          12       0.15      0.15      0.15       310
          13       0.00      0.00      0.00        21
          14       0.07      0.07      0.07       165

    accuracy                           0.09      2080
   macro avg       0.06      0.06      0.06      2080
weighted avg       0.09   

In [4]:

import tensorflow as tf
print(tf.__version__)
print(tf.__version__)
print(tf.test.is_built_with_cuda())       # Should be True
print(tf.config.list_physical_devices('GPU'))  # Should list your GPU

\
print("Built with CUDA:",tf.config.list_physical_devices('GPU'))
print("GPU available:", tf.test.is_gpu_available())

2.20.0
2.20.0
False
[]
Built with CUDA: []
Instructions for updating:
Use `tf.config.list_physical_devices('GPU')` instead.
GPU available: False


In [15]:
import tensorflow as tf
print("TensorFlow version:", tf.__version__)
print("Built with CUDA:", tf.test.is_built_with_cuda())
print("GPUs available:", tf.config.list_physical_devices('GPU'))


TensorFlow version: 2.20.0
Built with CUDA: False
GPUs available: []
